# 1. Cài đặt Thư viện và Khởi tạo
Cài đặt các công cụ cần thiết để xử lý ngôn ngữ tự nhiên và huấn luyện mô hình (PEFT/LoRA).

In [ ]:
!pip install -q nltk rouge-score bert-score pandas
!pip install -q -U peft trl datasets accelerate transformers
!pip install -q -U bitsandbytes>=0.46.1 accelerate
!pip install -q chromadb>=0.5.0 sentence-transformers langchain-chroma langchain-huggingface langchain-core

import os
import json
import torch
import pandas as pd
import re
from tqdm import tqdm
from datasets import Dataset

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
nltk.download('punkt', quiet=True)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

import chromadb
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 2. Thiết lập Môi trường và Tải Dữ liệu
Kết nối Google Drive và chuẩn bị bộ dữ liệu pháp luật dưới định dạng Alpaca prompt.

In [ ]:
# --- 1. THIẾT LẬP ĐƯỜNG DẪN ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
BASE_PATH = "/content/drive/MyDrive/NLP"

DATASET_DIR = os.path.join(BASE_PATH, "data/datasets")
CHECKPOINT_DIR = os.path.join(BASE_PATH, "checkpoints")
ADAPTER_FINAL_DIR = os.path.join(CHECKPOINT_DIR, "qwen_lora_law_production")
VECTOR_DB_DIR = os.path.join(BASE_PATH, "vector_db")

TRAIN_FILE = os.path.join(DATASET_DIR, "data_qa_train.json")
TEST_FILE = os.path.join(DATASET_DIR, "data_qa_test.json")

# Đặt HuggingFace Token
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"

# --- 2. TẢI DỮ LIỆU ĐÁNH GIÁ ---
def load_and_format_data(file_path):
    print(f"[*] Đang tải dữ liệu từ {file_path}...")
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    dataset = Dataset.from_list(data)
    dataset = dataset.map(lambda x: {"text": f"### Question:\n{x['question']}\n### Answer:\n{x['answer']}"})
    return dataset, data

train_dataset, _ = load_and_format_data(TRAIN_FILE)
eval_dataset, test_data = load_and_format_data(TEST_FILE)

# --- 3. TÁI TẠO RETRIEVER TỪ DRIVE ---
print("\n[*] Tái tạo Retriever từ Google Drive...")
class MyVietnameseEmbeddingFunction(chromadb.EmbeddingFunction):
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.model = SentenceTransformer(model_name)
    def __call__(self, input: list) -> list:
        return self.model.encode(input, normalize_embeddings=True).tolist()

class FocusedAnswerParser(StrOutputParser):
    def parse(self, text: str) -> str:
        text = text.strip()
        if "[TRẢ LỜI]:" in text: answer = text.split("[TRẢ LỜI]:")[-1].strip()
        else: answer = text
        answer = re.sub(r'^\s*[\-\*]\s*', '', answer, flags=re.MULTILINE)
        answer = re.sub(r'\n',' ', answer)
        lines = [line.strip() for line in answer.split('.') if line.strip() and len(line.strip()) > 5]
        return '. '.join(lines) + ('.' if lines else '')

# Khởi tạo lại ChromaDB thẳng từ ổ cứng
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
db = Chroma(collection_name="traffic_law_collection", embedding_function=embedding_model, persist_directory=VECTOR_DB_DIR)
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 5})
print("[+] Retriever đã sẵn sàng.")

Mounted at /content/drive
[*] Đang tải dữ liệu từ /content/drive/MyDrive/NLP/data/datasets/data_qa_train.json...


Map:   0%|          | 0/313 [00:00<?, ? examples/s]

[*] Đang tải dữ liệu từ /content/drive/MyDrive/NLP/data/datasets/data_qa_test.json...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]


[*] Tái tạo Retriever từ Google Drive...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[+] Retriever đã sẵn sàng.


# 3. Khởi tạo và Huấn luyện Mô hình (Fine-tuning)
Sử dụng Qwen2.5-3B với kỹ thuật LoRA và Quantization 4-bit để tối ưu bộ nhớ.

In [ ]:
model_id = "Qwen/Qwen2.5-3B-Instruct"

# Cấu hình Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"[*] Đang tải mô hình gốc {model_id} (Quantization 4-bit)...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

DO_TRAIN = True

if DO_TRAIN:
    print("\n[*] Cấu hình PEFT và chuẩn bị Model cho K-bit Training...")
    base_model.config.use_cache = False
    base_model_prepared = prepare_model_for_kbit_training(base_model)

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        optim="paged_adamw_32bit",
        save_strategy="epoch",
        eval_strategy="steps",
        eval_steps=10,
        logging_steps=10,
        learning_rate=2e-4,
        weight_decay=0.001,
        fp16=False,
        bf16=True,
        max_grad_norm=0.3,
        num_train_epochs=3,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=base_model_prepared,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
    )

    print("[*] Bắt đầu quá trình Fine-tuning (Vui lòng chờ...).")
    trainer.train()

    print(f"[*] Đang lưu mô hình Fine-tuned vào thư mục: {ADAPTER_FINAL_DIR}")
    trainer.model.save_pretrained(ADAPTER_FINAL_DIR)
    tokenizer.save_pretrained(ADAPTER_FINAL_DIR)
    print(f"[+] Lưu thành công.")

    finetuned_model = trainer.model

else:
    print(f"\n[*] Bỏ qua bước Training. Tải lại mô hình đã Fine-tune từ: {ADAPTER_FINAL_DIR}")
    if os.path.exists(ADAPTER_FINAL_DIR):
        finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_FINAL_DIR)
        print("[+] Đã kết hợp Base Model với LoRA Adapter thành công.")
    else:
        raise FileNotFoundError(f"[!] Không tìm thấy Adapter tại {ADAPTER_FINAL_DIR}. Vui lòng set DO_TRAIN = True.")

# --- TÁI TẠO RAG CHAIN TỪ MODEL VỪA LOAD ---
print("\n[*] Tái tạo RAG Chain để đánh giá...")
model_pipeline = pipeline(
    "text-generation", model=finetuned_model, tokenizer=tokenizer,
    temperature=0.1, max_new_tokens=450, pad_token_id=tokenizer.eos_token_id, do_sample=True, top_p=0.75
)
lc_llm = HuggingFacePipeline(pipeline=model_pipeline)

prompt = PromptTemplate.from_template("""Bạn là trợ lý AI pháp luật Việt Nam.
[TÀI LIỆU]:
{context}

[CÂU HỎI]:
{question}

Hãy trả lời dựa trên tài liệu. Nếu tài liệu không có thông tin, nói rõ "Không có thông tin".
Trả lời đầy đủ thông tin, không thêm bất kỳ chi tiết nào ngoài tài liệu.
[TRẢ LỜI]: """)

def format_docs(docs):
    formatted, seen = [], set()
    for doc in docs:
        content = doc.page_content.strip()
        if content and len(content) > 40 and content not in seen:
            formatted.append(f"[{doc.metadata.get('article', 'Luật')}]\n{content}")
            seen.add(content)
    return "\n\n".join(formatted)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | lc_llm | FocusedAnswerParser()
)
print("[+] RAG Chain đã sẵn sàng.")

[*] Đang tải mô hình gốc Qwen/Qwen2.5-3B-Instruct (Quantization 4-bit)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


[*] Cấu hình PEFT và chuẩn bị Model cho K-bit Training...


Adding EOS to train dataset:   0%|          | 0/313 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/313 [00:00<?, ? examples/s]

...

[*] Bắt đầu quá trình Fine-tuning (Vui lòng chờ...).


Step,Training Loss,Validation Loss
10,2.546892,2.019774
20,1.716776,1.556881
30,1.413326,1.433183
40,1.298894,1.363163
50,1.046355,1.333418
60,1.031180,1.275032
70,1.000960,1.221412
80,0.955952,1.180278
90,0.683078,1.178716
100,0.639033,1.193465


[*] Đang lưu mô hình Fine-tuned vào thư mục: /content/drive/MyDrive/NLP/checkpoints/qwen_lora_law_production


[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'temperature', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[+] Lưu thành công.

[*] Tái tạo RAG Chain để đánh giá...
[+] RAG Chain đã sẵn sàng.
